In [56]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from IPython.display import display

In [73]:
# Data prepration 
dataFrame = pd.read_csv('course_lead_scoring_2026_hw3.csv', delimiter=',')

# substitute NA / 0 for non existent values
numeric_cols = dataFrame.select_dtypes(include="number").columns
string_cols = dataFrame.select_dtypes(include=["object", "string"]).columns
display(numeric_cols)
display("null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display(string_cols)
display("null string columns :" + str(dataFrame[string_cols].isnull().sum()))
dataFrame[numeric_cols] = dataFrame[numeric_cols].fillna(0)
dataFrame[string_cols] = dataFrame[string_cols].fillna("NA")
display("after fixed null numeric columns :" + str(dataFrame[numeric_cols].isnull().sum()))
display("after fixed null string columns :" + str(dataFrame[string_cols].isnull().sum()))


Index(['annual_income', 'number_of_courses_viewed', 'interaction_count',
       'lead_score', 'converted'],
      dtype='str')

'null numeric columns :annual_income               369\nnumber_of_courses_viewed      0\ninteraction_count             0\nlead_score                   35\nconverted                     0\ndtype: int64'

Index(['lead_source', 'industry', 'employment_status', 'location'], dtype='str')

'null string columns :lead_source          148\nindustry             240\nemployment_status    194\nlocation             208\ndtype: int64'

'after fixed null numeric columns :annual_income               0\nnumber_of_courses_viewed    0\ninteraction_count           0\nlead_score                  0\nconverted                   0\ndtype: int64'

'after fixed null string columns :lead_source          0\nindustry             0\nemployment_status    0\nlocation             0\ndtype: int64'

In [ ]:
# 1. most frequent observation (mode) for the column industry
industry_mode = dataFrame['industry'].mode()[0]
display("most frequent observation for the column industry: " + str(industry_mode))

# Split the data
df_full_train, df_test = train_test_split(dataFrame, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)


'most frequent observation for the column industry: technology'

In [77]:
#2. Biggest correlation
correlation_matrix = dataFrame[numeric_cols].corr()
display(correlation_matrix)
abs_correlations = correlation_matrix.abs().to_numpy(copy=True)

# Exclude self-correlations.
np.fill_diagonal(abs_correlations, np.nan)

#find the largest correlation index in the frame
max_corr_idx = np.unravel_index(
    np.nanargmax(abs_correlations), abs_correlations.shape
)
max_corr_value = abs_correlations[max_corr_idx]
display("Maximum correlation value: " + str(max_corr_value))

display(
    f"Biggest absolute correlation is between "
    f"{correlation_matrix.columns[max_corr_idx[0]]} and "
    f"{correlation_matrix.columns[max_corr_idx[1]]}: {max_corr_value:.4f}"
)

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


'Maximum correlation value: 0.9157457980418697'

'Biggest absolute correlation is between interaction_count and lead_score: 0.9157'

In [79]:
#3. Biggest MI (Mutual Information)
#Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
#Round the scores to 2 decimals using round(score, 2).
from sklearn.metrics import mutual_info_score

y_train = df_train["converted"]
X_train = df_train.drop(columns=["converted"])

categorical_cols = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns

mi_scores = {
    col: mutual_info_score(y_train, X_train[col])
    for col in categorical_cols
}

display(
    sorted(
        [(col, round(score, 2)) for col, score in mi_scores.items()],
        key=lambda item: item[1],
        reverse=True,
    )
)

[('lead_source', 0.03),
 ('employment_status', 0.02),
 ('industry', 0.0),
 ('location', 0.0)]